# BackPaS — setup reproducible en Colab

Este notebook restaura el entorno desde Drive y no inicia entrenamientos automáticamente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, tarfile, glob, subprocess
REPO='/content/backbone-pbo'
DRIVE_ROOT='/content/drive/MyDrive/PBO-Tesis'
JOB=None  # Define aquí solo una técnica pendiente cuando corresponda
SOURCE_ARCHIVE=f'{DRIVE_ROOT}/runs/pbo-tesis-source.tar.gz'
if os.path.exists(SOURCE_ARCHIVE) and not os.path.exists(f'{REPO}/src'):
    os.makedirs(REPO, exist_ok=True)
    with tarfile.open(SOURCE_ARCHIVE,'r:gz') as tar: tar.extractall(REPO)
if not os.path.exists(f'{REPO}/src'):
    subprocess.run(['git','clone','https://github.com/jgomezaTI/PBO-DA.git',REPO],check=True)


In [ ]:
DATA_ROOT=f'{DRIVE_ROOT}/datasets'
for name in ['MVC','CA']:
    archive=f'{DATA_ROOT}/{name}/{name}.tar.gz'
    if os.path.exists(archive) and not os.path.exists(f'{REPO}/data/backpas/{name}'):
        with tarfile.open(archive,'r:gz') as tar: tar.extractall(f'{REPO}/data/backpas')
mis_dir=f'{DATA_ROOT}/MIS'
mis_archive=f'{mis_dir}/MIS.tar.gz'
parts=sorted(glob.glob(f'{mis_dir}/MIS.tar.gz.part-*'))
if not os.path.exists(mis_archive) and parts:
    with open(mis_archive,'wb') as out:
        for part in parts:
            with open(part,'rb') as inp: shutil.copyfileobj(inp,out)
if os.path.exists(mis_archive) and not os.path.exists(f'{REPO}/data/backpas/MIS'):
    with tarfile.open(mis_archive,'r:gz') as tar: tar.extractall(f'{REPO}/data/backpas')
results_archive=f'{DRIVE_ROOT}/runs/training-results.tar.gz'
if os.path.exists(results_archive):
    with tarfile.open(results_archive,'r:gz') as tar: tar.extractall(REPO)
subprocess.run(['python','-m','pip','install','-e','.[ml]'],cwd=REPO,check=True)
print('Setup completado. Baselines existentes no se ejecutan.')


In [ ]:
if JOB is None:
    print('Define JOB con una técnica pendiente para ejecutar entrenamiento.')
else:
    subprocess.run(['python','scripts/cloud_run.py','--repo',REPO,'--job',JOB,'--drive-root',DRIVE_ROOT,'--resume'],cwd=REPO,check=True)
